# AIND Ephys Spike Sorting (Kilosort4)

Build an `AINDEPhysSpikesortKilosort4ScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDEPhysSpikesortKilosort4Task` for each coordinate. The task itself clones [aind-ephys-spikesort-kilosort4](https://github.com/AllenNeuralDynamics/aind-ephys-spikesort-kilosort4) on first use, seeds its `data/` with the `preprocessed_<name>/` and `binary_<name>.json` produced by `AINDEPhysPreprocessingTask` in notebook 02, writes a `params_obi.json`, runs `code/run_capsule.py`, and copies the resulting `spikesorted_<name>/` into the single config's `coordinate_output_root` (`obi-output/03_aind_ephys_spikesort_kilosort4/grid_scan/0/`).

Notebook 02 already preprocesses 9 s of the recording, which is long enough for KS4 to fit whitening / templates — so this notebook does **no** preprocessing of its own. We just tune a few sorter knobs (`whitening_range`, `nskip`, `nearest_chans`, `nearest_templates`, `do_correction=False`, `nblocks=0`, `torch_device="cpu"`) for the small toy probe.

## Imports and prerequisites

In [1]:
import subprocess
import sys
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.aind_ephys._03_kilosort4.blocks import Kilosort4Sorter

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "kilosort", "aind-data-schema", "scikit-learn",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv


Resolved 44 packages in 445ms
Uninstalled 3 packages in 49ms
Installed 3 packages in 18ms
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2
 - setuptools==82.0.0
 + setuptools==81.0.0


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'kilosort', 'aind-data-schema', 'scikit-learn'], returncode=0)

## Build the spike-sorting scan config

`preprocessing_output_path` reads notebook 02's preprocessed binary directly from its `coordinate_output_root`.

In [3]:
preprocessing_output_path = (
    Path.cwd() / "../../../obi-output/02_aind_ephys_preprocessing/grid_scan/0"
).resolve()
assert preprocessing_output_path.exists(), (
    f"{preprocessing_output_path} not found. Run 02_aind_ephys_preprocessing.ipynb first."
)

ks_scan = obi.AINDEPhysSpikesortKilosort4ScanConfig(
    initialize=obi.AINDEPhysSpikesortKilosort4ScanConfig.Initialize(
        preprocessing_output_path=preprocessing_output_path,
        skip_motion_correction=True,
        min_drift_channels=64,
        n_jobs=1,
    ),
    sorter=Kilosort4Sorter(
        do_correction=False,
        nblocks=0,
        torch_device="cpu",
        whitening_range=16,
        nskip=1,
        nearest_chans=8,
        nearest_templates=16,
    ),
)

## Generate the grid scan and run the sorting task

In [4]:
ks_grid = obi.GridScanGenerationTask(
    form=ks_scan,
    output_root="../../../obi-output/03_aind_ephys_spikesort_kilosort4/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
ks_grid.execute()

obi.run_tasks_for_generated_scan(ks_grid)

[2026-04-29 19:02:39,088] INFO: Seeded 1 preprocessed recording(s) into /tmp/aind-ephys-spikesort-kilosort4/data


[2026-04-29 19:02:39,088] INFO: Running python -u run_capsule.py --params params_obi.json --n-jobs 1 --min-drift-channels 64 --skip-motion-correction --raise-if-fails


[2026-04-29 19:02:57,471] INFO: 

SPIKE SORTING WITH KILOSORT4

	RAISE_IF_FAILS: True
	SKIP_MOTION_CORRECTION: True
	MIN_DRIFT_CHANNELS: 64
	N_JOBS: -1
Sorting recording: block0_None_recording1
Loading recording from binary JSON
BinaryFolderRecording: 70 channels - 30.0kHz - 1 segments - 270,001 samples - 9.00s 
                       float32 dtype - 72.10 MiB
Drift correction disabled
	Raw sorting output: KiloSortSortingExtractor: 14 units - 1 segments - 30.0kHz
	Sorting output without empty units: UnitsSelectionSorting: 14 units - 1 segments - 30.0kHz
	Saving results to ../results/spikesorted_block0_None_recording1
SPIKE SORTING time: 17.38s



[PosixPath('../../../obi-output/03_aind_ephys_spikesort_kilosort4/grid_scan/0')]

## Load the sorting result

In [5]:
import spikeinterface.full as si

coord_dir = Path(ks_grid.single_configs[0].coordinate_output_root)
print("coordinate_output_root:", coord_dir)

sorting_dirs = [
    p for p in coord_dir.iterdir() if p.is_dir() and p.name.startswith("spikesorted_")
]
print("Sorting dirs:", [p.name for p in sorting_dirs])

if sorting_dirs:
    sorting = si.load(sorting_dirs[0])
    print(sorting)
    print("num_units:", len(sorting.unit_ids))
    print("unit_ids:", list(sorting.unit_ids))

coordinate_output_root: ../../../obi-output/03_aind_ephys_spikesort_kilosort4/grid_scan/0
Sorting dirs: ['spikesorted_block0_None_recording1']
NumpyFolder (NumpyFolderSorting): 14 units - 1 segments - 30.0kHz
num_units: 14
unit_ids: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13)]
